# Discord Trivia Bot -- Question Generation & Scoring Demo (No Discord)

This notebook is a companion to the course project
[Build a Discord Trivia Bot](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/trivia-bot).
It demos the two non-Discord pieces of that project -- generating a fresh trivia question on a topic with a
free-tier LLM, and tracking per-player scores -- with plain function calls and a few fake "players" answering,
the exact same `round.py`/`scores.py`/`generate.py` logic the real bot uses in `examples/trivia-bot/`.

It does **not** run the actual Discord bot: a live bot holds an open connection to Discord's Gateway and needs
to keep running indefinitely, which a hosted notebook runtime like this one can't provide (see the lesson's
"Where to run this" section for why). For the real, live bot, see `bot.py` and run it locally or in a GitHub
Codespace as described in the lesson.

## 1. Install dependencies

No `discord.py` needed here -- just the LLM client.

In [ ]:
!pip install -q openai

## 2. The fixed question bank

The exact same bank as `questions.py` in the real project, used when no topic is given.

In [ ]:
import random

QUESTION_BANK = [
    {
        "question": "What year was Python first released?",
        "options": ["1989", "1991", "1995", "2000"],
        "answer_index": 1,
    },
    {
        "question": "Which planet is known as the Red Planet?",
        "options": ["Venus", "Jupiter", "Mars", "Saturn"],
        "answer_index": 2,
    },
    {
        "question": "What is the largest ocean on Earth?",
        "options": ["Atlantic", "Indian", "Arctic", "Pacific"],
        "answer_index": 3,
    },
]


def random_question():
    return random.choice(QUESTION_BANK)


random_question()

## 3. Format and check questions

Identical logic to `round.py`: render a question as text, and check whether a submitted letter is correct.
Neither function needs Discord or an LLM at all.

In [ ]:
OPTION_LETTERS = "ABCD"


def format_question(question):
    lines = [f"**{question['question']}**"]
    for letter, option in zip(OPTION_LETTERS, question["options"]):
        lines.append(f"{letter}) {option}")
    return "\n".join(lines)


def check_answer(question, letter):
    letter = letter.strip().upper()
    valid_letters = OPTION_LETTERS[: len(question["options"])]
    if letter not in valid_letters:
        return False
    return valid_letters.index(letter) == question["answer_index"]


q = random_question()
print(format_question(q))
print("Correct guess?", check_answer(q, OPTION_LETTERS[q["answer_index"]]))
print("Wrong guess?", check_answer(q, OPTION_LETTERS[(q["answer_index"] + 1) % 4]))

## 4. Get a free-tier LLM API key

Question generation needs one free-tier LLM key -- same options as the lesson's table. **GitHub Models** is
the suggested default (a GitHub personal access token with the `models: read` scope, from
[github.com/settings/tokens](https://github.com/settings/tokens), no separate signup). Gemini, Groq, Mistral,
Cerebras, and OpenRouter all work too -- see the lesson for details on each.

The cell below uses `getpass` so the key is never echoed to the notebook's output or saved in it.

In [ ]:
from getpass import getpass

llm_api_key = getpass("Paste your GitHub Models token (or other provider key): ")

## 5. Generate a fresh trivia question on a topic

Identical logic to `generate.py`'s `generate_question()`: ask the LLM for a multiple-choice question about a
topic as JSON, then validate the shape of what comes back -- an LLM asked for JSON can still return something
malformed. This block defaults to GitHub Models' OpenAI-compatible endpoint; swap the `base_url` (and the key
above) for your own provider's client if you picked a different one.

In [ ]:
import json

from openai import OpenAI

llm_client = OpenAI(
    api_key=llm_api_key,
    base_url="https://models.github.ai/inference",  # swap for your provider's base_url if different
)

PROMPT_TEMPLATE = """Write one multiple-choice trivia question about: {topic}

Respond with ONLY a JSON object, no other text, in exactly this shape:
{{"question": "...", "options": ["...", "...", "...", "..."], "answer_index": 0}}

Requirements:
- Exactly 4 options.
- Exactly one is correct; put its index (0-3) in answer_index.
- The wrong options must be plausible, not obviously silly.
- Keep the question and every option short enough to fit in a Discord message."""


def generate_question(topic):
    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",  # confirm this still has a free tier before running
        messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(topic=topic)}],
        response_format={"type": "json_object"},
    )
    question = json.loads(response.choices[0].message.content)

    options = question.get("options")
    answer_index = question.get("answer_index")
    if not question.get("question") or not isinstance(options, list) or len(options) != 4:
        raise ValueError(f"LLM returned a malformed question: {question!r}")
    if not isinstance(answer_index, int) or not (0 <= answer_index < 4):
        raise ValueError(f"LLM returned an invalid answer_index: {question!r}")
    return question


generated = generate_question("classic video games")
print(format_question(generated))

## 6. Score tracking, with fake players

Identical logic to `scores.py`: an in-memory dict standing in for `scores.json` here (the real project writes
it to disk so scores survive a bot restart), keyed by a fake Discord user id, with each player's latest
display name and running point total.

In [ ]:
def award_point(scores, user_id, display_name):
    key = str(user_id)
    entry = scores.get(key, {"name": display_name, "score": 0})
    entry["name"] = display_name
    entry["score"] += 1
    scores[key] = entry
    return scores


def leaderboard_text(scores, top_n=10):
    if not scores:
        return "No scores yet -- play a round with `/trivia`!"
    ranked = sorted(scores.values(), key=lambda entry: entry["score"], reverse=True)
    lines = [f"{i}. {entry['name']} — {entry['score']}" for i, entry in enumerate(ranked[:top_n], start=1)]
    return "\n".join(lines)


scores = {}
scores = award_point(scores, user_id=111, display_name="Alice")
scores = award_point(scores, user_id=222, display_name="Bob")
scores = award_point(scores, user_id=111, display_name="Alice")
print(leaderboard_text(scores))

## 7. Try it end to end: a few fake players answering a few rounds

Simulates what `bot.py`'s round loop does for each `/trivia` call -- pick a question (bank or a generated
topic), have a "player" answer, score it if correct -- without any Discord connection, timers, or real users
involved.

In [ ]:
fake_players = [(111, "Alice"), (222, "Bob"), (333, "Priya")]
topics = [None, "world capitals", "classic video games"]  # None = draw from the fixed bank

scores = {}
for topic, (player_id, player_name) in zip(topics, fake_players):
    question = generate_question(topic) if topic else random_question()
    print(format_question(question))
    print(f"({player_name} answers with the correct letter)")

    letter = OPTION_LETTERS[question["answer_index"]]  # simulate this fake player always answering correctly
    if check_answer(question, letter):
        scores = award_point(scores, player_id, player_name)
        print(f"✅ {player_name} scores!")
    print()

print("Final leaderboard:")
print(leaderboard_text(scores))

## What this notebook did (and didn't) show

This covered both non-Discord halves of the project: generating a fresh, validated trivia question on any
topic with a free-tier LLM, and tracking per-player scores across rounds -- exactly the logic `bot.py` calls
from inside its `/trivia` and `/leaderboard` slash commands.

What it did not show is the Discord layer itself: connecting to Discord's Gateway, staying online indefinitely,
posting a question in a real channel, collecting real players' replies within a time limit, and persisting
`scores.json` across restarts. That needs a genuine long-running process -- see the lesson's ["Where to run
this"](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/trivia-bot#where-to-run-this)
section for why Colab/Kaggle can't host that, and run `bot.py` locally (or in a GitHub Codespace) for the real,
live bot.